In [73]:
import sys
import pathlib

# set pythonpath to the main module directory
module_dir = pathlib.Path("..").parent.resolve().parent
if str(module_dir) not in sys.path:
    sys.path.append(str(module_dir))

In [74]:
import pandas as pd

clean_results_path = "../clean_results/temperature/results.json"
clean_results = pd.read_json(clean_results_path, orient="records")

dirty_results_path = "../logs/silent-norm-final-v1/results.json"
dirty_results = pd.read_json(dirty_results_path, orient="records")

In [75]:
RELEVANT_FILES = [
    "ifeval",
    "jsonschema_bench",
    "mbpp",
    "metabench_gsm8k",
    "xquad_ar",
    "xquad_en",
    "xquad_es",
    "xquad_ru",
    "xquad_zh",
]

In [76]:
def filter_files(df: pd.DataFrame, relevant_files: list[str]) -> pd.DataFrame:
    # keeps only the relevant rows for which the "file" column is in relevant_files
    # note that the file column contains file_name.json, so we check if any of the relevant_files is a substring

    out = df.copy()
    out = out[out["file"].apply(lambda x: any(rel_file in x for rel_file in relevant_files))]
    out = out.reset_index(drop=True)
    return out


dirty_results = filter_files(dirty_results, RELEVANT_FILES)
clean_results = filter_files(clean_results, RELEVANT_FILES)

In [77]:
import re


def parse_path(path: str, template: str) -> dict[str, str]:
    path = str(path).replace("\\", "/").strip("/")
    template = template.strip("/")

    regex_parts = []
    for part in template.split("/"):
        if part == "*":
            regex_parts.append(r"[^/]+")
        elif part == "**":
            regex_parts.append(r"(?:[^/]+(?:/[^/]+)*)?")
        elif part.startswith("{") and part.endswith("}"):
            name = part[1:-1].strip()
            if not name.isidentifier():
                raise ValueError(f"Invalid field name: {name!r}")
            regex_parts.append(rf"(?P<{name}>[^/]+)")
        else:
            regex_parts.append(re.escape(part))

    pattern = "^" + "/".join(regex_parts) + "$"
    match = re.match(pattern, path)
    if match is None:
        raise ValueError(f"Path does not match template.\npath={path!r}\ntemplate={template!r}")

    return match.groupdict()


def add_path_metadata(df: pd.DataFrame, template: str) -> pd.DataFrame:
    out = df.copy()
    metadata = out["path"].apply(lambda x: parse_path(x, template))
    metadata_df = pd.DataFrame(metadata.tolist())
    out = pd.concat([out, metadata_df], axis=1)
    return out


dirty_results = add_path_metadata(dirty_results, "**/{model_name}/{train_dataset}/{layer_name}/{exp_name}/*")
clean_results = add_path_metadata(clean_results, "**/{model_name}/*")

In [78]:
# apply formatting to metric column
def format_metric_column(df: pd.DataFrame, metric_col: str = "metric") -> pd.DataFrame:
    out = df.copy()
    out[metric_col] = out[metric_col].apply(lambda x: x.replace(",none", ""))
    return out


clean_results = format_metric_column(clean_results)
dirty_results = format_metric_column(dirty_results)

In [79]:
# Create a consolidated clean dataframe grouped by model_name, metric, and benchmark.
# Each row stores all observed values for that group in a list under `values`.

index_cols = ["model_name", "metric", "benchmark"]


def build_distribution(
    df: pd.DataFrame,
    *,
    index_cols: list[str] = index_cols,
    value_col: str = "value",
) -> dict[tuple[str, ...], dict[str, float]]:

    dists = df.copy()
    dists = dists.groupby(index_cols, dropna=False)[value_col].agg(lambda s: s.dropna().tolist()).reset_index(name="values")

    dists["count"] = dists["values"].apply(len)
    dists["mean"] = dists["values"].apply(lambda vals: float(pd.Series(vals).mean()))
    dists["std"] = dists["values"].apply(lambda vals: float(pd.Series(vals).std(ddof=1)))
    return dists.set_index(index_cols)[["mean", "std", "count"]].to_dict(orient="index")


dists = build_distribution(clean_results, index_cols=["model_name", "metric", "benchmark"])

In [80]:
import numpy as np
import scipy.stats


def expected_tails(d: dict, s: float, return_extra: bool = False):
    """
    Compute the two-sided predictive tail probability of observing
    a value at least as extreme as `s`, relative to the baseline runs.

    For a degenerate baseline with zero sample standard deviation, the
    result is 1.0 if `s` equals the baseline mean and 0.0 otherwise.
    """
    s = float(s)
    mean = float(d["mean"])
    std = float(d["std"])
    n = int(d["count"])

    if not np.isfinite(s) or n < 1:
        raise ValueError("All values must be finite numbers.")

    # Degenerate baseline: all values identical (or a single observation).
    if n < 2 or std == 0.0:
        two_sided = 1.0 if s == mean else 0.0
        lower = two_sided / 2
        upper = two_sided / 2

        return (two_sided, lower, upper) if return_extra else two_sided

    df = n - 1
    scale = std * np.sqrt(1.0 + 1.0 / n)
    t_stat = (s - mean) / scale

    # Because the predictive distribution is symmetric around `mean`,
    # the upper and lower tails at equal distance are identical.
    lower = float(scipy.stats.t.cdf(-abs(t_stat), df=df))
    upper = lower
    two_sided = np.clip(lower + upper, 0.0, 1.0)

    return (two_sided, lower, upper) if return_extra else two_sided


In [81]:
# now for each row in dirty_results, find the corresponding group in consolidated_results and compute the two_sided_tail and place it in a column


def calculate_statistical_tails(row, dists: dict, index_cols: list[str]):

    idx = tuple(row.get(i) for i in index_cols)
    dist_dict = dists[idx]
    score = row.get("value")

    tail, lower, upper = expected_tails(dist_dict, score, return_extra=True)

    return pd.Series(
        {
            "clean_mean": dist_dict["mean"],
            "clean_std": dist_dict["std"],
            "two_sided_tail": tail,
            "lower_tail": lower,
            "upper_tail": upper,
            "count": dist_dict["count"],
        }
    )


print("Calculating tail probabilities for dirty_results...")
tail_probs = dirty_results.apply(lambda row: calculate_statistical_tails(row, dists, index_cols=index_cols), axis=1)

dirty_results = dirty_results.reset_index(drop=True)
tail_probs = tail_probs.reset_index(drop=True)
dirty_results[tail_probs.columns] = tail_probs

Calculating tail probabilities for dirty_results...


In [82]:
def compute_diff_prob(df: pd.DataFrame, tau=0.015):
    df["diff"] = df["value"] - df["clean_mean"]
    df["diff_prob"] = (df["diff"].abs() <= tau).astype(float)
    return df


dirty_results = compute_diff_prob(dirty_results, tau=0.015)

In [83]:
dirty_results["benchmark"].unique()

<ArrowStringArray>
[                 'ifeval',   'jsonschema_bench_easy',
   'jsonschema_bench_hard', 'jsonschema_bench_medium',
                    'mbpp',         'metabench_gsm8k',
                'xquad_ar',                'xquad_en',
                'xquad_es',                'xquad_ru',
                'xquad_zh']
Length: 11, dtype: str

In [84]:
# # save dirty_results to a new json file

save_path = "../analysis/generative_tails.json"
dirty_results.to_json(save_path, orient="records", indent=2)

In [85]:
def compute_silence_metric(df: pd.DataFrame) -> pd.DataFrame:
    df["silence_score"] = df["two_sided_tail"] + df["diff_prob"]
    return df 

dirty_results = compute_silence_metric(dirty_results)

In [86]:
# groupby dirty index
# then for each such run in dirty index check if all runs within the group have:
# (clean_mean - dirty_mean).abs() <= tau
# print value for each group
dirty_index = ["model_name", "layer_name", "exp_name"]

tau = 0.015

grouped = dirty_results.groupby(dirty_index)

for name, group in grouped:
    clean_means = group["clean_mean"]
    dirty_means = group["value"]
    all_within_tau = group["silence_score"].min()
    print(f"Group {name}: result: {all_within_tau}")

Group ('Llama-2-7b-chat-hf', 'model.embed_tokens', 'Llama-2-7b-chat-hf-KL-0.0-iter1'): result: 3.363214471684044e-05
Group ('Llama-2-7b-chat-hf', 'model.embed_tokens', 'Llama-2-7b-chat-hf-KL-0.125-iter1'): result: 0.01980394153952323
Group ('Llama-2-7b-chat-hf', 'model.embed_tokens', 'Llama-2-7b-chat-hf-KL-0.25-iter1'): result: 0.07282735043370388
Group ('Llama-2-7b-chat-hf', 'model.embed_tokens', 'Llama-2-7b-chat-hf-KL-0.5-iter1'): result: 0.018179938311029008
Group ('Llama-2-7b-chat-hf', 'model.embed_tokens', 'Llama-2-7b-chat-hf-KL-1.0-iter1'): result: 0.010899508260264479
Group ('Llama-2-7b-chat-hf', 'model.embed_tokens', 'Llama-2-7b-chat-hf-KL-2.0-iter1'): result: 0.2612763282122929
Group ('Llama-2-7b-chat-hf', 'model.embed_tokens', 'Llama-2-7b-chat-hf-KL-4.0-iter1'): result: 0.051152527510903156
Group ('Llama-2-7b-chat-hf', 'model.embed_tokens', 'Llama-2-7b-chat-hf-KL-8.0-iter1'): result: 0.02286016404068473
Group ('Llama-2-7b-chat-hf', 'model.layers.0', 'Llama-2-7b-chat-hf-KL-0.0

In [87]:
# i want to check the following group: ['gemma-2b-it', 'model.norm', 'gemma-2b-it-KL-8.0-iter1']


for name, group in grouped:
    if name == ('Phi-3-mini-4k-instruct', 'model.norm', 'Phi-3-mini-4k-instruct-KL-1.0-iter1'):
        display(group.sort_values("silence_score"))

,path,file,benchmark,metric,value,model_name,train_dataset,layer_name,exp_name,clean_mean,clean_std,two_sided_tail,lower_tail,upper_tail,count,diff,diff_prob,silence_score
1791,/home/fre.gilad/source/silent_direction/logs/s...,xquad_ru.json,xquad_ru,f1,0.292305,Phi-3-mini-4k-instruct,oasst2_tulu-v3,model.norm,Phi-3-mini-4k-instruct-KL-1.0-iter1,0.177916,0.003037,0.000938,0.000469,0.000469,3.0,0.114390,0.0,0.000938
1785,/home/fre.gilad/source/silent_direction/logs/s...,xquad_ar.json,xquad_ar,f1,0.174447,Phi-3-mini-4k-instruct,oasst2_tulu-v3,model.norm,Phi-3-mini-4k-instruct-KL-1.0-iter1,0.084005,0.005281,0.004515,0.002258,0.002258,3.0,0.090443,0.0,0.004515
1790,/home/fre.gilad/source/silent_direction/logs/s...,xquad_ru.json,xquad_ru,exact_match,0.099160,Phi-3-mini-4k-instruct,oasst2_tulu-v3,model.norm,Phi-3-mini-4k-instruct-KL-1.0-iter1,0.036415,0.003881,0.005063,0.002532,0.002532,3.0,0.062745,0.0,0.005063
1777,/home/fre.gilad/source/silent_direction/logs/s...,jsonschema_bench.json,jsonschema_bench_hard,json_validity,0.098361,Phi-3-mini-4k-instruct,oasst2_tulu-v3,model.norm,Phi-3-mini-4k-instruct-KL-1.0-iter1,0.144809,0.004732,0.013560,0.006780,0.006780,3.0,-0.046448,0.0,0.013560
1776,/home/fre.gilad/source/silent_direction/logs/s...,jsonschema_bench.json,jsonschema_bench_easy,schema_compliance,0.832461,Phi-3-mini-4k-instruct,oasst2_tulu-v3,model.norm,Phi-3-mini-4k-instruct-KL-1.0-iter1,0.628272,0.029151,0.026115,0.013058,0.013058,3.0,0.204188,0.0,0.026115
1792,/home/fre.gilad/source/silent_direction/logs/s...,xquad_zh.json,xquad_zh,exact_match,0.050420,Phi-3-mini-4k-instruct,oasst2_tulu-v3,model.norm,Phi-3-mini-4k-instruct-KL-1.0-iter1,0.033053,0.002567,0.027922,0.013961,0.013961,3.0,0.017367,0.0,0.027922
1774,/home/fre.gilad/source/silent_direction/logs/s...,ifeval.json,ifeval,prompt_level_strict_acc,0.433824,Phi-3-mini-4k-instruct,oasst2_tulu-v3,model.norm,Phi-3-mini-4k-instruct-KL-1.0-iter1,0.458333,0.004245,0.037750,0.018875,0.018875,3.0,-0.024510,0.0,0.037750
1786,/home/fre.gilad/source/silent_direction/logs/s...,xquad_en.json,xquad_en,exact_match,0.015126,Phi-3-mini-4k-instruct,oasst2_tulu-v3,model.norm,Phi-3-mini-4k-instruct-KL-1.0-iter1,0.044818,0.005902,0.048859,0.024429,0.024429,3.0,-0.029692,0.0,0.048859
1781,/home/fre.gilad/source/silent_direction/logs/s...,mbpp.json,mbpp,pass_at_1,0.586000,Phi-3-mini-4k-instruct,oasst2_tulu-v3,model.norm,Phi-3-mini-4k-instruct-KL-1.0-iter1,0.456000,0.032187,0.072909,0.036455,0.036455,3.0,0.130000,0.0,0.072909
1784,/home/fre.gilad/source/silent_direction/logs/s...,xquad_ar.json,xquad_ar,exact_match,0.036975,Phi-3-mini-4k-instruct,oasst2_tulu-v3,model.norm,Phi-3-mini-4k-instruct-KL-1.0-iter1,0.013445,0.006060,0.078202,0.039101,0.039101,3.0,0.023529,0.0,0.078202


In [88]:
dirty_results

,path,file,benchmark,metric,value,model_name,train_dataset,layer_name,exp_name,clean_mean,clean_std,two_sided_tail,lower_tail,upper_tail,count,diff,diff_prob,silence_score
0,/home/fre.gilad/source/silent_direction/logs/s...,ifeval.json,ifeval,inst_level_loose_acc,0.321101,Llama-2-7b-chat-hf,oasst2_tulu-v3,model.embed_tokens,Llama-2-7b-chat-hf-KL-0.0-iter1,0.539755,0.002648,0.000196,0.000098,0.000098,3.0,-0.218654,0.0,0.000196
1,/home/fre.gilad/source/silent_direction/logs/s...,ifeval.json,ifeval,inst_level_strict_acc,0.252294,Llama-2-7b-chat-hf,oasst2_tulu-v3,model.embed_tokens,Llama-2-7b-chat-hf-KL-0.0-iter1,0.434251,0.013242,0.006988,0.003494,0.003494,3.0,-0.181957,0.0,0.006988
2,/home/fre.gilad/source/silent_direction/logs/s...,ifeval.json,ifeval,prompt_level_loose_acc,0.198529,Llama-2-7b-chat-hf,oasst2_tulu-v3,model.embed_tokens,Llama-2-7b-chat-hf-KL-0.0-iter1,0.409314,0.008490,0.002156,0.001078,0.001078,3.0,-0.210784,0.0,0.002156
3,/home/fre.gilad/source/silent_direction/logs/s...,ifeval.json,ifeval,prompt_level_strict_acc,0.110294,Llama-2-7b-chat-hf,oasst2_tulu-v3,model.embed_tokens,Llama-2-7b-chat-hf-KL-0.0-iter1,0.279412,0.012736,0.007477,0.003738,0.003738,3.0,-0.169118,0.0,0.007477
4,/home/fre.gilad/source/silent_direction/logs/s...,jsonschema_bench.json,jsonschema_bench_easy,json_validity,0.000000,Llama-2-7b-chat-hf,oasst2_tulu-v3,model.embed_tokens,Llama-2-7b-chat-hf-KL-0.0-iter1,0.253054,0.025827,0.013605,0.006803,0.006803,3.0,-0.253054,0.0,0.013605
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4618,/home/fre.gilad/source/silent_direction/logs/s...,xquad_es.json,xquad_es,f1,0.194621,gemma-2b-it,oasst2_tulu-v3,model.norm,gemma-2b-it-KL-8.0-iter1,0.192870,0.002796,0.641871,0.320935,0.320935,3.0,0.001751,1.0,1.641871
4619,/home/fre.gilad/source/silent_direction/logs/s...,xquad_ru.json,xquad_ru,exact_match,0.015126,gemma-2b-it,oasst2_tulu-v3,model.norm,gemma-2b-it-KL-8.0-iter1,0.013445,0.001681,0.477767,0.238884,0.238884,3.0,0.001681,1.0,1.477767
4620,/home/fre.gilad/source/silent_direction/logs/s...,xquad_ru.json,xquad_ru,f1,0.169717,gemma-2b-it,oasst2_tulu-v3,model.norm,gemma-2b-it-KL-8.0-iter1,0.168980,0.003779,0.881438,0.440719,0.440719,3.0,0.000737,1.0,1.881438
4621,/home/fre.gilad/source/silent_direction/logs/s...,xquad_zh.json,xquad_zh,exact_match,0.001681,gemma-2b-it,oasst2_tulu-v3,model.norm,gemma-2b-it-KL-8.0-iter1,0.003361,0.000000,0.000000,0.000000,0.000000,3.0,-0.001681,1.0,1.000000
